In [2]:
import json
import re
from ff3 import FF3Cipher

# ==========================================
# ทำ FPE (FF3/FF3-1) กับข้อมูลจริงจากไฟล์ OCSF log (ocsf.log)
# เป้าหมาย: Tokenize field ที่อ่อนไหว เช่น hostname, ip, username, agent uid
# โดยยังคงรูปแบบ [XXX_NN] เดิมไว้ (Format-Preserving)
# ==========================================

# 128-bit key (32 hex chars) และ tweak (8 byte / 16 hex chars) สำหรับ FF3-1
# ⚠️ ในระบบจริงต้องดึงจาก KMS/Environment Variable ห้าม Hardcode
FF3_KEY = "F3D2FE0E66B2AA56806944AA2770D53A"
FF3_TWEAK = "6A81B1DC1D04B6CA"

# Field ในไฟล์นี้เป็น placeholder รูปแบบ [ตัวอักษรพิมพ์ใหญ่/ตัวเลข/underscore]
# เช่น [HOST_01], [INT_IP_01], [USER_01], [AGENT_01]
# จึงต้องสร้าง Alphabet เอง (FF3-1 รองรับ custom alphabet ต่างจาก FF3 เดิม)
ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_"
field_cipher = FF3Cipher.withCustomAlphabet(FF3_KEY, FF3_TWEAK, ALPHABET)

PLACEHOLDER_RE = re.compile(r"\[[A-Za-z0-9_]+\]")


def tokenize_field(cipher: FF3Cipher, value: str) -> str:
    """Tokenize ข้อความภายในวงเล็บ [] แล้วคืนค่าในรูปแบบ [] เดิม"""
    inner = value[1:-1]
    return f"[{cipher.encrypt(inner)}]"


def walk_and_tokenize(node, cipher: FF3Cipher):
    """เดิน JSON แบบ recursive แล้ว Tokenize ทุก field ที่อยู่ในรูปแบบ [XXX_NN]"""
    if isinstance(node, dict):
        return {key: walk_and_tokenize(val, cipher) for key, val in node.items()}
    if isinstance(node, list):
        return [walk_and_tokenize(val, cipher) for val in node]
    if isinstance(node, str) and PLACEHOLDER_RE.fullmatch(node):
        return tokenize_field(cipher, node)
    return node


with open("../Data/ocsf.log", "r", encoding="utf-8") as f:
    ocsf_log = json.load(f)

tokenized_log = walk_and_tokenize(ocsf_log, field_cipher)

print("--- ก่อน Tokenize ---")
print(f"device.hostname        : {ocsf_log['device']['hostname']}")
print(f"device.ip              : {ocsf_log['device']['ip']}")
print(f"device.agent_list[0].uid: {ocsf_log['device']['agent_list'][0]['uid']}")
print(f"evidences[0].user.name : {ocsf_log['evidences'][0]['user']['name']}")

print("\n--- หลัง Tokenize (FF3-1, รูปแบบ [XXX] เดิม) ---")
print(f"device.hostname        : {tokenized_log['device']['hostname']}")
print(f"device.ip              : {tokenized_log['device']['ip']}")
print(f"device.agent_list[0].uid: {tokenized_log['device']['agent_list'][0]['uid']}")
print(f"evidences[0].user.name : {tokenized_log['evidences'][0]['user']['name']}")

# ตรวจสอบว่าสามารถถอดรหัสกลับเป็นค่าเดิมได้ (Reversible)
recovered_hostname = f"[{field_cipher.decrypt(tokenized_log['device']['hostname'][1:-1])}]"
assert recovered_hostname == ocsf_log['device']['hostname']
print(f"\nถอดรหัส device.hostname กลับ: {recovered_hostname}")


--- ก่อน Tokenize ---
device.hostname        : [HOST_01]
device.ip              : [INT_IP_01]
device.agent_list[0].uid: [AGENT_01]
evidences[0].user.name : [USER_01]

--- หลัง Tokenize (FF3-1, รูปแบบ [XXX] เดิม) ---
device.hostname        : [5SDAH7D]
device.ip              : [4LN51D49F]
device.agent_list[0].uid: [858NYC0W]
evidences[0].user.name : [9K_MUDC]

ถอดรหัส device.hostname กลับ: [HOST_01]
